In [145]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
words = open('names.txt', 'r').read().splitlines()

# Standardize: 

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi["."] = 0  
itos = {i:s for s,i in stoi.items()}

vocab_size = len(stoi)

In [156]:
def build_dataset(data, block_size): 

    X, Y = [], []
    #block_size = 3 
    for word in data: 
        #print(word)
        context = [0] * block_size

        for char in word + ".": 
            
            #print(f"Current Context: {''.join(itos[i] for i in context)}, -> {context}")
            ix = stoi[char]
            #print(f"Next character is: '{char}', String to number: {char} -> {xi}")

            X.append(context)
            Y.append(ix)   

            #print(f"X to Y mapping: {context}, -> {ix}")
            # update context
            context = context[1:] + [ix]

            #print(''.join(itos[i] for i in context), '----->', itos[ix], "\n") 

    
    return(torch.tensor(X), torch.tensor(Y))


In [171]:
block_size = 3
emb_dim = 2
hidden_size = 100

g = torch.Generator().manual_seed(21)  # reproducibility

# Create dataset
X, Y = build_dataset(data=words, block_size=block_size)
print(X.shape, Y.shape)

# Embedding table
C = torch.randn((vocab_size, emb_dim), generator=g)
print(C.shape)

# First layer: flattened embeddings -> hidden layer
W1 = torch.randn((emb_dim * block_size, hidden_size), generator=g)
b1 = torch.randn(hidden_size, generator=g)
print(W1.shape, b1.shape)

# Output layer: hidden layer -> vocabulary logits
W2 = torch.randn((hidden_size, vocab_size), generator=g)
b2 = torch.randn(vocab_size, generator=g)
print(W2.shape, b2.shape)

parameters = [C, W1, b1, W2, b2]
# Required setup step 
for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

torch.Size([228146, 3]) torch.Size([228146])
torch.Size([27, 2])
torch.Size([6, 100]) torch.Size([100])
torch.Size([100, 27]) torch.Size([27])


3481

In [ ]:

print(X_batch)

tensor([[ 0, 13,  1],
        [ 1,  9,  1],
        [ 0, 11,  9],
        [ 0,  0, 11],
        [19, 20, 15],
        [13, 25, 12],
        [14, 25, 11],
        [18, 19,  5],
        [13,  5,  4],
        [ 5, 12, 25],
        [ 3,  5,  5],
        [22,  1, 10],
        [ 3,  8,  1],
        [ 0,  0,  9],
        [ 0,  0,  0],
        [ 0,  9,  7],
        [ 0, 13,  1],
        [ 0, 14, 15],
        [ 0,  0,  1],
        [12,  5, 20],
        [ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  5],
        [ 4, 23,  1],
        [ 1, 20,  1],
        [18,  1, 14],
        [15, 12, 21],
        [ 0,  0,  0],
        [ 0,  1, 21],
        [ 1, 26, 26],
        [ 0,  8, 21],
        [13,  5, 18],
        [ 0,  0, 13],
        [21, 13,  1],
        [14,  5, 14],
        [12,  5,  9],
        [ 1, 14,  9],
        [ 0,  1,  8],
        [ 0,  0,  0],
        [ 5, 12, 12],
        [ 1,  3,  9],
        [ 0,  0,  0],
        [14,  5, 19],
        [15, 18,  5],
        [ 0,  7, 18],
        [ 

In [218]:
for _ in range(100): 

    # Sample
    batch = torch.randint(0, X.shape[0], (100,))
    X_batch = X[batch,] ; Y_batch = Y[batch]

    # forward pass
    emb = C[X_batch]
    h = torch.tanh(emb.view(-1, emb_dim * block_size) @ W1 + b1)
    logits = h @ W2 + b2 
    loss = F.cross_entropy(logits, Y_batch)
    #print(loss.item())

    # backward pass 
    for p in parameters: 
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad


print(loss.item())


49.22132873535156
